# DFFN deblurring: Colab backend
Runs the three frozen experts of the current **v5** model (HINet, SFNet, NAFNet with TLC) plus the trained
**DFFN** fusion module (the 0.97M gated fuser from the DFFN notebooks) behind a small FastAPI server and exposes it through a public tunnel.
The website only sends one image and shows the outputs. **Deblurring only.**

Run the cells top to bottom on a GPU runtime (T4 is enough, L4/A100 is faster).
Before this notebook you need, in your Google Drive `MyDrive/DFFN/`:
* the `weights` folder with `HINet-GoPro.pth`, the SFNet GoPro `.pkl` and **`NAFNet-GoPro-width64.pth`**
  (the first two are from Step 2 / 2b of the DFFN notebook; NAFNet is downloaded for you if missing), and
* a trained **v5** deblurring DFFN checkpoint (HINet + SFNet + NAFNet experts), saved by the v5 notebook.
  A checkpoint trained with AdaIR as the third expert will NOT load here (this notebook tells you if so).
  Without it the DFFN card shows a placeholder that equals plain averaging, and the page says so.

## 1. Setup

In [ ]:
import os, sys, re, json, glob, io, time, base64, shutil, subprocess, threading, math
import torch
assert torch.cuda.is_available(), "Switch to a GPU runtime: Runtime > Change runtime type > T4 / L4 / A100 GPU"
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "einops", "gdown", "addict", "yapf", "lmdb",
                "pytorch_msssim", "fastapi", "uvicorn", "python-multipart"], check=True)

## 2. CONFIG: Drive, weights, serving
* `ROOT` is the same Drive folder the DFFN notebook writes to.
* `FUSION_CKPT` can point at one specific trained deblurring checkpoint. Leave it empty and the newest
  `*deblur*dffn*.pt` found under `ROOT` is used.
* `NAFNET_TAP` is the layer whose input is NAFNet's feature map for the fuser. The default (`ending`, 64 channels) is the
  analogue of HINet's `last`; the notebook checks it against your checkpoint and tells you if the width differs.

In [ ]:
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/DFFN"
FUSION_CKPT = ""          # e.g. "/content/drive/MyDrive/DFFN/results/v5/ckpt/deblur_DFFN.pt"
NAFNET_TAP = "ending"     # NAFNet layer whose INPUT feeds the fuser (64 ch for the width-64 model)
MAX_SIDE = 1024           # longest image side processed (bigger uploads are downscaled to fit the GPU)
USE_TTA = True            # horizontal-flip ensemble for DFFN, as in the DFFN notebook (about 2x slower)
PORT = 8000
CODE = "/content"
print("NOTEBOOK VERSION: dffn-deblur-v10-nafnet (2026-11-26)")

## 3. Code and weights (same layout as the DFFN notebook)

In [ ]:
def sh(cmd):
    print("$", cmd); return subprocess.run(cmd, shell=True, check=False)

os.chdir(CODE)
for name, url in {"HINet": "https://github.com/megvii-model/HINet", "SFNet": "https://github.com/c-yn/SFNet",
                  "NAFNet": "https://github.com/megvii-research/NAFNet"}.items():
    if not os.path.isdir(f"{CODE}/{name}"):
        sh(f"git clone -q --depth 1 {url} {CODE}/{name}")
vp = f"{CODE}/HINet/basicsr/version.py"       # normally generated by setup.py
if not os.path.exists(vp):
    open(vp, "w").write("__version__ = '1.2.0'\nshort_version = '1.2.0'\nversion_info = (1, 2, 0)\n")

def locate_weights():
    for pat in ["/content/drive/MyDrive/*/weights", "/content/drive/MyDrive/*/*/weights", "/content/drive/MyDrive/weights"]:
        for d in sorted(glob.glob(pat)):
            if glob.glob(f"{d}/**/HINet-GoPro.pth", recursive=True):
                return d
    return None

W = locate_weights()
if not W:                                      # same downloads as Step 2b of the DFFN notebook
    W = f"{ROOT}/weights"; os.makedirs(W, exist_ok=True)
    os.chdir(W)
    if not os.path.exists("HINet-GoPro.pth"): sh("gdown 1dw8PKVkLfISzNtUu3gqGh83NBO83ZQ5n")
    if not os.path.isdir("sfnet_gopro"): sh("gdown 1mVerQce1ZZFkKOj0Cbwyj49ZPWdHcO2z && unzip -q GoPro-SFNet.zip -d sfnet_gopro")
    os.chdir(CODE)
# NAFNet GoPro width-64 weights (33.71 dB): look in the weights folder, else try the official Drive file
if not glob.glob(f"{W}/**/NAFNet-GoPro-width64.pth", recursive=True):
    sh(f"gdown 1S0PVRbyTakYY9a82kujgZLbMihfNBLfC -O {W}/NAFNet-GoPro-width64.pth")   # official NAFNet README link
    if not os.path.exists(f"{W}/NAFNet-GoPro-width64.pth") or os.path.getsize(f"{W}/NAFNet-GoPro-width64.pth") < 1_000_000:
        print(">>> Could not download NAFNet-GoPro-width64.pth. Download it from the NAFNet README (GoPro, width64) and put it in", W)
print("weights:", W)
for f in sorted(glob.glob(f"{W}/**/*", recursive=True)):
    if os.path.isfile(f) and f.endswith((".pth", ".pkl", ".ckpt")): print(f"   {os.path.relpath(f, W):45s} {os.path.getsize(f)/1e6:7.0f} MB")

# SFNet's own model folder, copied as a package exactly like the DFFN notebook does
if not os.path.isdir(f"{CODE}/sfnet_deblur"):
    shutil.copytree(f"{CODE}/SFNet/Motion_deblurring/models", f"{CODE}/sfnet_deblur")
    open(f"{CODE}/sfnet_deblur/__init__.py", "a").close()

# NAFNet's architecture as a standalone package. Its repo also calls itself `basicsr`, which would clash with HINet's,
# so the three files we need are copied and their imports made relative.
if not os.path.isdir(f"{CODE}/nafnet_arch"):
    os.makedirs(f"{CODE}/nafnet_arch")
    A = f"{CODE}/NAFNet/basicsr/models/archs"
    for fn in ["NAFNet_arch.py", "local_arch.py", "arch_util.py"]:
        t = open(f"{A}/{fn}").read()
        t = t.replace("from basicsr.models.archs.arch_util import", "from .arch_util import").replace("from basicsr.models.archs.local_arch import", "from .local_arch import")
        t = t.replace("from basicsr.utils import get_root_logger", "get_root_logger = lambda *a, **k: None")
        open(f"{CODE}/nafnet_arch/{fn}", "w").write(t)
    open(f"{CODE}/nafnet_arch/__init__.py", "w").close()
for p in [CODE, f"{CODE}/HINet", f"{CODE}/SFNet/pytorch-gradual-warmup-lr"]:
    if p not in sys.path: sys.path.insert(0, p)

## 4. The three frozen experts
Same taps as the DFFN notebook: HINet = input of `last` (64 ch), SFNet = `Decoder[2]` output (32 ch),
NAFNet = input of `ending` (64 ch). SFNet uses its full-image `('test','GOPRO')` mode and NAFNet its
test-time local converter (`NAFNetLocal`, TLC), as in the report.

In [ ]:
import numpy as np
import torch.nn as nn, torch.nn.functional as F
dev = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False        # uploads have arbitrary sizes; benchmarking every new size would be slow

def tload(path): return torch.load(path, map_location="cpu", weights_only=False)
def strip_module(sd): return {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}
def find(*patterns):
    for pattern in patterns:
        hits = sorted(glob.glob(f"{W}/**/{pattern}", recursive=True))
        if hits: return hits[0]
    raise FileNotFoundError(f"Missing weight file {patterns} in {W}")

class Expert:
    def __init__(self, model, layer_fn, use_input):
        self.model = model.eval().to(dev)
        for p in self.model.parameters(): p.requires_grad_(False)
        self.feat = None
        def hook(mod, inp, out): self.feat = inp[0] if use_input else out
        layer_fn(self.model).register_forward_hook(hook)
    @torch.no_grad()
    def __call__(self, x):
        y = self.model(x)
        if isinstance(y, (list, tuple)): y = y[-1]
        return y, self.feat

def load_hinet():
    from basicsr.models.archs.hinet_arch import HINet
    m = HINet(wf=64, hin_position_left=3, hin_position_right=4)
    m.load_state_dict(tload(find("HINet-GoPro.pth"))["params"], strict=True)
    return Expert(m, lambda n: n.last, True)

def load_sfnet():
    from sfnet_deblur.SFNet import build_net
    m = build_net(("test", "GOPRO"))
    m.load_state_dict(tload(find("GOPRO.pkl", "sfnet_gopro/**/*.pkl"))["model"], strict=True)
    return Expert(m, lambda n: n.Decoder[2], False)

def load_nafnet():
    from nafnet_arch.NAFNet_arch import NAFNetLocal       # NAFNet with TLC (test-time local converter)
    m = NAFNetLocal(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1, dec_blk_nums=[1, 1, 1, 1])
    ck = tload(find("NAFNet-GoPro-width64.pth"))
    m.load_state_dict(strip_module(ck.get("params", ck)), strict=True)
    return Expert(m, lambda n: dict(n.named_modules())[NAFNET_TAP], True)

ORDER = ["hinet", "sfnet", "nafnet"]
DISPLAY = {"hinet": "HINet", "sfnet": "SFNet", "nafnet": "NAFNet", "average": "Output averaging", "fusion": "DFFN (ours)"}
EXPERTS, LOAD_NOTES = {}, {}
for key, loader in zip(ORDER, [load_hinet, load_sfnet, load_nafnet]):
    try:
        EXPERTS[key] = loader(); print(f"[ok]   {key}")
    except Exception as e:
        EXPERTS[key] = None; LOAD_NOTES[key] = f"{type(e).__name__}: {str(e)[:200]}"; print(f"[SKIP] {key}: {LOAD_NOTES[key]}")

## 5. DFFN fusion module
The current DFFN (0.97M parameters): the gate also sees the image, the three outputs and their disagreement and has a
large receptive field. Class and layer names are exactly those used in training, so the checkpoint loads with `strict=True`.
An untrained module is exactly a plain average of the three outputs, which is what the page shows as a placeholder.

In [ ]:
class SE(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c // r, 1), nn.ReLU(inplace=True),
                                nn.Conv2d(c // r, c, 1), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(c, c, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(c, c, 3, padding=1))
    def forward(self, x): return x + self.body(x)

class DFFN(nn.Module):
    """DFFN fuser, verbatim from the DFFN notebooks (use_gate=True)."""
    def __init__(self, in_chs=(64, 32, 96), c=64, n_blocks=6, use_gate=True):
        super().__init__()
        self.use_gate = use_gate
        n = len(in_chs)
        self.align = nn.ModuleList([nn.Conv2d(ci, c, 1) for ci in in_chs])
        self.img_emb = nn.Sequential(nn.Conv2d(3 * (1 + 2 * n), c, 3, padding=1), nn.ReLU(inplace=True),
                                     nn.Conv2d(c, c, 3, padding=1))
        if use_gate:
            self.g_in = nn.Conv2d((n + 1) * c, c, 1)
            self.g_enc = ResBlock(c)
            self.g_d1 = nn.Sequential(nn.Conv2d(c, c, 3, stride=2, padding=1), nn.ReLU(inplace=True), ResBlock(c))
            self.g_d2 = nn.Sequential(nn.Conv2d(c, c, 3, stride=2, padding=1), nn.ReLU(inplace=True), ResBlock(c), ResBlock(c))
            self.g_ctx = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c, 1), nn.ReLU(inplace=True))
            self.g_u1 = nn.Conv2d(c, c, 3, padding=1)
            self.g_u0 = nn.Conv2d(c, c, 3, padding=1)
            self.g_out = nn.Conv2d(c, n, 3, padding=1)
            nn.init.zeros_(self.g_out.weight); nn.init.zeros_(self.g_out.bias)
        self.se = SE(c)
        self.refine = nn.Sequential(*[ResBlock(c) for _ in range(n_blocks)])
        self.head = nn.Conv2d(c, 3, 3, padding=1)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)
    def forward(self, feats, outs, x):
        a = [f(t) for f, t in zip(self.align, feats)]
        mean = sum(outs) / len(outs)
        emb = self.img_emb(torch.cat([x] + list(outs) + [o - mean for o in outs], 1))
        if self.use_gate:
            g0 = self.g_enc(self.g_in(torch.cat(a + [emb], 1)))
            g1 = self.g_d1(g0)
            g2 = self.g_d2(g1)
            g2 = g2 + self.g_ctx(g2)
            g1 = self.g_u1(g1 + F.interpolate(g2, size=g1.shape[-2:], mode="bilinear", align_corners=False))
            g0 = self.g_u0(g0 + F.interpolate(g1, size=g0.shape[-2:], mode="bilinear", align_corners=False))
            w = torch.softmax(self.g_out(g0), dim=1)
        else:
            b, _, h, wd = a[0].shape
            w = torch.full((b, len(a), h, wd), 1.0 / len(a), device=a[0].device)
        fused = sum(w[:, i:i + 1] * a[i] for i in range(len(a))) + emb
        fused = self.refine(self.se(fused))
        ensemble = sum(w[:, i:i + 1] * outs[i] for i in range(len(outs)))
        return ensemble + self.head(fused), w

def find_fusion_ckpt():
    if FUSION_CKPT: return FUSION_CKPT
    cands = [p for p in glob.glob(f"{ROOT}/**/*.pt", recursive=True) + glob.glob("/content/weights/fusion_deblur.pth")
             if "deblur" in os.path.basename(p).lower() and "dffn" in os.path.basename(p).lower() and "nogate" not in p.lower()]
    return max(cands, key=os.path.getmtime) if cands else ""

def build_fusion(path):
    if not path:
        return DFFN().to(dev).eval(), False, "none (placeholder = plain average)"
    ck = tload(path)
    sd = strip_module(ck.get("model", ck.get("params", ck.get("state_dict", ck))))
    if "g_in.weight" not in sd:
        raise RuntimeError(f"{os.path.basename(path)} is not a gated DFFN checkpoint (no g_in.weight). Old 0.24M checkpoints are not supported here.")
    in_chs = tuple(sd[f"align.{i}.weight"].shape[1] for i in range(3))
    live = tuple(EXPERT_CHANNELS)
    if in_chs != live:
        raise RuntimeError(f"Feature widths differ: checkpoint expects {in_chs} (HINet, SFNet, NAFNet) but the experts here give {live}. "
                           "If the third differs, set NAFNET_TAP to the layer your v5 notebook used; if the checkpoint was trained with AdaIR "
                           "(third width 96) it cannot be used with NAFNet.")
    c = sd["align.0.weight"].shape[0]
    n_blocks = len({k.split(".")[1] for k in sd if k.startswith("refine.")})
    net = DFFN(in_chs, c, n_blocks)
    net.load_state_dict(sd, strict=True)
    return net.to(dev).eval(), True, f"DFFN (image-aware gate, c={c}, {n_blocks} blocks)"

FUSION, FUSION_TRAINED, FUSION_ARCH, FUSION_PATH, EXPERT_CHANNELS = None, False, "", find_fusion_ckpt(), []
if all(EXPERTS.values()):
    with torch.no_grad():                                   # dry run to read the real feature widths
        _x = torch.rand(1, 3, 64, 64, device=dev)
        EXPERT_CHANNELS = [EXPERTS[k](_x)[1].shape[1] for k in ORDER]
    print("expert feature widths (HINet, SFNet, NAFNet):", EXPERT_CHANNELS)
    try:
        FUSION, FUSION_TRAINED, FUSION_ARCH = build_fusion(FUSION_PATH)
        n_par = sum(p.numel() for p in FUSION.parameters()) / 1e6
        print(f"DFFN: {FUSION_ARCH} | {n_par:.3f}M parameters | trained={FUSION_TRAINED}")
        print("checkpoint:", FUSION_PATH or "NOT FOUND: the DFFN card will be a placeholder (plain average)")
    except Exception as e:
        print("\n>>> DFFN checkpoint NOT loaded:", e)
        FUSION, FUSION_TRAINED, FUSION_ARCH = DFFN(tuple(EXPERT_CHANNELS)).to(dev).eval(), False, "none (placeholder = plain average)"
else:
    print("DFFN unavailable: not all three experts loaded:", LOAD_NOTES)

## 6. Pipeline and metrics
Scores use the DFFN notebook's own `score` (outputs rounded to 8 bit, PSNR from MSE, SSIM from `pytorch_msssim`), RGB for deblurring.

In [ ]:
from PIL import Image
from pytorch_msssim import ssim as ssim_fn

def load_rgb(data: bytes, max_side=MAX_SIDE, size=None):
    im = Image.open(io.BytesIO(data)).convert("RGB")
    if size is not None:
        if im.size != size: im = im.resize(size, Image.LANCZOS)
    else:
        s = max(im.size)
        if s > max_side:
            r = max_side / s; im = im.resize((max(32, round(im.width * r)), max(32, round(im.height * r))), Image.LANCZOS)
    return im

def run_experts(x):
    """x: [1,3,h,w] in 0..1. Returns per-expert outputs/features (cropped back to h x w), timings and notes."""
    h, w = x.shape[-2:]
    H, Wd = math.ceil(h / 32) * 32, math.ceil(w / 32) * 32
    mode = "reflect" if (H - h < h and Wd - w < w) else "replicate"
    xp = F.pad(x, (0, Wd - w, 0, H - h), mode=mode) if (H, Wd) != (h, w) else x
    outs, feats, ms, notes = {}, {}, {}, {}
    for key in ORDER:
        e = EXPERTS[key]
        if e is None: notes[key] = "Not loaded: " + LOAD_NOTES.get(key, "unknown"); continue
        try:
            torch.cuda.synchronize(); t = time.time()
            y, f = e(xp); torch.cuda.synchronize()
            outs[key], feats[key], ms[key] = y[..., :h, :w].clamp(0, 1), f[..., :h, :w], round((time.time() - t) * 1000)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); notes[key] = "Out of GPU memory. Try a smaller image."
        except Exception as ex:
            notes[key] = f"{type(ex).__name__}: {str(ex)[:160]}"
    return outs, feats, ms, notes

def to_u8(t):  # [1,3,H,W] -> HxWx3 uint8
    return (t.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)

def data_url(u8):
    buf = io.BytesIO(); Image.fromarray(u8).save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def score(pred, gt):  # both [1,3,H,W] in 0..1, RGB
    p, g = (pred.clamp(0, 1) * 255).round() / 255, (gt * 255).round() / 255
    mse = F.mse_loss(p, g).item()
    try: s = ssim_fn(p, g, data_range=1.0, size_average=True).item()
    except Exception: s = None          # images smaller than the SSIM window
    return 10 * math.log10(1.0 / max(mse, 1e-10)), s

TRUST_COLORS = {"hinet": (59, 130, 246), "sfnet": (245, 158, 11), "nafnet": (217, 70, 239)}   # same as the website legend

def make_trust(wmap, base):
    """Colour every pixel by the experts DFFN trusts there, over a greyscale copy of the output."""
    cols = torch.tensor([TRUST_COLORS[k] for k in ORDER], device=wmap.device, dtype=torch.float32).view(1, 3, 3, 1, 1) / 255.0
    col = (wmap[:, :, None] * cols).sum(1)
    gray = base.mean(1, keepdim=True).expand(-1, 3, -1, -1)
    ov = 0.35 * gray + 0.65 * col
    share = wmap.mean(dim=(0, 2, 3)).tolist()
    return {"image": data_url(to_u8(ov)), "share": {k: round(100 * v, 1) for k, v in zip(ORDER, share)}}

def blank(key): return {"id": key, "name": DISPLAY[key], "available": False, "note": "", "image": None, "psnr": None, "ssim": None, "time_ms": None}

@torch.inference_mode()
def run_pipeline(image_bytes, gt_bytes=None):
    t_all = time.time()
    im = load_rgb(image_bytes)
    w, h = im.size
    x = torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)[None].to(dev)
    gt = None
    if gt_bytes: gt = torch.from_numpy(np.asarray(load_rgb(gt_bytes, size=im.size), dtype=np.float32) / 255.0).permute(2, 0, 1)[None].to(dev)

    outs, feats, ms, notes = run_experts(x)
    items, imgs, trust = [], {}, None
    for key in ORDER:
        it = blank(key)
        if key in outs: it.update(available=True, time_ms=ms[key]); imgs[key] = outs[key]
        else: it["note"] = notes.get(key, "No output")
        items.append(it)

    avg = blank("average")
    if outs:
        used = [k for k in ORDER if k in outs]
        imgs["average"] = torch.stack([outs[k] for k in used]).mean(0)
        avg.update(available=True, time_ms=0)
        if len(used) < 3: avg["note"] = "Mean of " + ", ".join(DISPLAY[k] for k in used) + " (the others are unavailable)."
    else: avg["note"] = "No expert produced an output."
    items.append(avg)

    fus = blank("fusion")
    if FUSION is not None and len(outs) == 3:
        try:
            torch.cuda.synchronize(); t = time.time()
            pred, wmap = FUSION([feats[k] for k in ORDER], [outs[k] for k in ORDER], x)
            if USE_TTA:                                   # flip ensemble, exactly as in the DFFN notebook's evaluation
                xf = x.flip(-1); o2, f2, _, _ = run_experts(xf)
                if len(o2) == 3: pred = (pred + FUSION([f2[k] for k in ORDER], [o2[k] for k in ORDER], xf)[0].flip(-1)) / 2
            torch.cuda.synchronize()
            pred = pred.clamp(0, 1); imgs["fusion"] = pred
            trust = make_trust(wmap, pred)
            fus.update(available=True, time_ms=round((time.time() - t) * 1000))
            if not FUSION_TRAINED: fus["note"] = "Placeholder: no trained DFFN checkpoint found, so this equals plain output averaging."
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); fus["note"] = "Out of GPU memory. Try a smaller image."
        except Exception as ex:
            fus["note"] = f"{type(ex).__name__}: {str(ex)[:160]}"
    else:
        fus["note"] = "DFFN needs all three experts loaded. " + "; ".join(f"{DISPLAY[k]}: {v}" for k, v in notes.items())
    items.append(fus)

    for it in items:
        if it["id"] in imgs:
            it["image"] = data_url(to_u8(imgs[it["id"]]))
            if gt is not None: it["psnr"], it["ssim"] = score(imgs[it["id"]], gt)
    out = {"task": "deblur", "width": w, "height": h, "y_channel": False, "has_gt": gt is not None,
           "fusion_trained": bool(FUSION_TRAINED and FUSION is not None), "trust": trust, "tta": USE_TTA,
           "elapsed_ms": round((time.time() - t_all) * 1000), "input": data_url(np.asarray(im)), "results": items}
    del outs, feats, imgs; torch.cuda.empty_cache()
    return out

## 7. API server
`GET /health` and `POST /restore` (multipart: `image`, optional `gt`; `task` must be `deblur` if sent).

In [ ]:
from typing import Optional
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="DFFN deblurring API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
gpu_lock = threading.Lock()

@app.get("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.get_device_name(0), "max_side": MAX_SIDE, "tta": USE_TTA,
            "models": {"deblur": {k: EXPERTS[k] is not None for k in ORDER}},
            "fusion_trained": {"deblur": bool(FUSION_TRAINED)}, "fusion_arch": FUSION_ARCH}

@app.post("/restore")
def restore(image: UploadFile = File(...), task: str = Form("deblur"), gt: Optional[UploadFile] = File(None)):
    if task != "deblur": raise HTTPException(400, "This backend only supports task 'deblur'")
    try:
        img_bytes = image.file.read(); gt_bytes = gt.file.read() if gt is not None else None
        with gpu_lock:
            return run_pipeline(img_bytes, gt_bytes)
    except HTTPException: raise
    except Exception as e:
        raise HTTPException(500, f"{type(e).__name__}: {e}")

## 8. Smoke test (random image, runs the whole pipeline once and warms the GPU)

In [ ]:
_buf = io.BytesIO(); Image.fromarray((np.random.rand(256, 320, 3) * 255).astype(np.uint8)).save(_buf, "PNG")
try:
    _r = run_pipeline(_buf.getvalue(), _buf.getvalue())
    print("pipeline OK in", _r["elapsed_ms"], "ms | DFFN trained:", _r["fusion_trained"])
    for it in _r["results"]: print(f"  {it['name']:18s} available={it['available']}  time={it['time_ms']} ms  {it['note']}")
except Exception:
    import traceback; traceback.print_exc()
    print("\nSmoke test failed (see above). The server below still starts, but fix this first.")

## 9. Start server and public tunnel
Copy the printed `https://....trycloudflare.com` URL into the website's "Backend URL" box.
Keep this notebook running while you demo. Quick tunnels close requests after about 100 s.

In [ ]:
import uvicorn
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)

if not os.path.exists(f"{CODE}/cloudflared"):
    sh(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CODE}/cloudflared && chmod +x {CODE}/cloudflared")
tun = subprocess.Popen([f"{CODE}/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: public_url = m.group(0); break
print("\nBACKEND URL  ->", public_url)
threading.Thread(target=lambda: [None for _ in tun.stdout], daemon=True).start()   # keep draining the tunnel log